# Stage 7 — Reporting (McKinsey Standard)
## Glassdoor Data-Science Jobs · Tier A pipeline

This notebook assembles `reports/final_report.html` — the single deliverable a stakeholder reads. It
regenerates every exhibit from the processed data and `_key_figures.json` (nothing is hardcoded),
embeds each as a base64 PNG, and writes one self-contained HTML file with all CSS inline and no
external requests, styled strictly to `DOCS/DESIGN.md`.

**The story, locked from evidence:**
- **SCR → Governing Thought → 4 MECE Key Lines → recommendation**, all on page 1 (the One-Page Test).
- Key Lines map to issue-tree branches **A (artifact), A/B (unmodelable), C (real structure),
  D (shippable model)**.

**Governs:** `DOCS/STRUCTURE.md` §7 · `DOCS/DESIGN.md` · `IMPLEMENTATION_PLAN.md` §8

In [1]:
from __future__ import annotations

import base64
import io
import json
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
PROC, REPORTS = ROOT / "data" / "processed", ROOT / "reports"

NAVY, BLUE, TEAL = "#051C2C", "#2251FF", "#00857C"
AMBER, SLATE, GREY, GRID = "#C1841C", "#7F93A6", "#9FADB8", "#E9ECEF"
CATEGORICAL = [BLUE, TEAL, AMBER, SLATE, GREY]
mpl.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica Neue", "DejaVu Sans"],
    "text.color": NAVY, "axes.labelcolor": NAVY, "xtick.color": SLATE, "ytick.color": SLATE,
    "axes.edgecolor": GRID, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.7, "axes.axisbelow": True,
    "axes.titlelocation": "left", "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlecolor": NAVY, "axes.titlepad": 10, "axes.labelsize": 9.5,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.frameon": False, "legend.fontsize": 9,
    "figure.dpi": 120, "savefig.bbox": "tight",
})

clean = pd.read_parquet(PROC / "postings_clean.parquet")
dedup = pd.read_parquet(PROC / "postings_dedup.parquet")
K = json.loads((REPORTS / "_key_figures.json").read_text())

def fig_to_b64(fig) -> str:
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

print(f"loaded {len(clean)} rows, {len(K)} key figures")
print(f"governing thought inputs: eta2_salary={K['eta2_salary_batch']}, "
      f"multi-band={K['pct_multi_band']}%, best R2={K['salary_best_grouped_r2']}, "
      f"tagger F1={K['tagger_macro_f1']}")

loaded 672 rows, 55 key figures
governing thought inputs: eta2_salary=1.0, multi-band=98.7%, best R2=-0.008, tagger F1=0.708


## 7.1 Regenerate the exhibits

Eight exhibits, one per Key Line plus the hero. Each returns a base64 string embedded into the HTML.

In [2]:
EX = {}

# EX1 (HERO) - contiguous blocks
fig, ax = plt.subplots(figsize=(9.5, 4))
ax.scatter(clean["index"], clean.salary_avg, s=10, color=BLUE, alpha=0.85, linewidths=0)
for b in clean.batch_id.unique():
    s = clean[clean.batch_id == b]
    ax.plot([s["index"].min(), s["index"].max()], [s.salary_avg.iloc[0]] * 2,
            color=NAVY, lw=1.7, solid_capstyle="butt")
ax.set_title("30 salary values, 30 unbroken blocks of rows — the field indexes the scrape")
ax.set_xlabel("row index (original scrape order)"); ax.set_ylabel("salary midpoint ($K)")
ax.grid(axis="x", visible=False)
EX["hero"] = fig_to_b64(fig)

# EX2 - repeated postings disagree
rep = clean[clean.duplicated("Job Description", keep=False)]
sp = rep.groupby("Job Description").agg(lo=("salary_avg", "min"), hi=("salary_avg", "max"),
                                        co=("company_clean", "first"), ti=("Job Title", "first"))
sp["spread"] = sp.hi - sp.lo
top = sp.nlargest(12, "spread").sort_values("spread")
fig, ax = plt.subplots(figsize=(9.5, 4.4))
yv = np.arange(len(top))
ax.hlines(yv, top.lo, top.hi, color=GREY, lw=2.4)
ax.scatter(top.lo, yv, s=40, color=SLATE, label="lowest band")
ax.scatter(top.hi, yv, s=40, color=BLUE, label="highest band")
ax.set_yticks(yv); ax.set_yticklabels([f"{r.co[:22]} — {r.ti[:26]}" for r in top.itertuples()], fontsize=7.5)
ax.set_xlabel("salary midpoint ($K) on the SAME posting"); ax.legend(loc="lower right")
ax.set_title(f"One posting, many prices — up to ${K['max_spread_k']:.0f}K apart")
ax.grid(axis="y", visible=False)
EX["repeat"] = fig_to_b64(fig)
print("EX1, EX2 done")

EX1, EX2 done


In [3]:
# EX3 - within-batch control
def eta_sq(v, g):
    v = pd.Series(v).astype(float); ok = v.notna(); v, g = v[ok], pd.Series(g)[ok]
    gm = v.mean(); ssb = sum(len(x) * (x.mean() - gm) ** 2 for _, x in v.groupby(g))
    return ssb / ((v - gm) ** 2).sum()
ctrl = {"salary_avg": clean.salary_avg, "Rating": clean.Rating,
        "desc_word_count": clean.desc_word_count, "company_age": clean.company_age}
fig, axes = plt.subplots(1, 4, figsize=(12, 3.1))
for ax, (nm, sv) in zip(axes, ctrl.items()):
    z = (sv - sv.mean()) / sv.std()
    col = BLUE if nm == "salary_avg" else GREY
    for b in range(clean.batch_id.nunique()):
        vals = z[clean.batch_id == b]
        ax.scatter([b] * len(vals), vals, s=4, color=col, alpha=0.5, linewidths=0)
    ax.set_title(f"{nm}\nη²={eta_sq(sv, clean.batch_id):.3f}", fontsize=9.5)
    ax.set_xlabel("batch"); ax.set_ylim(-3.2, 3.6); ax.grid(axis="x", visible=False)
    (ax.set_ylabel("z-score") if nm == "salary_avg" else ax.set_yticks([]))
EX["control"] = fig_to_b64(fig)

# EX4 - salary bake-off
sm = pd.read_csv(REPORTS / "_salary_models.csv").sort_values("R2_grouped")
fig, ax = plt.subplots(figsize=(9.5, 4))
yv = np.arange(len(sm))
ax.barh(yv - 0.2, sm.R2_random, height=0.38, color=GREY, label="random split (leaky)")
ax.barh(yv + 0.2, sm.R2_grouped, height=0.38, color=BLUE, label="grouped by posting (honest)")
ax.axvline(0, color=NAVY, lw=1.1)
ax.axvline(0.20, color=AMBER, ls="--", lw=1.6)
ax.text(0.205, len(sm) - 0.6, "acceptance\nbar R²=0.20", color=AMBER, fontsize=8.5, va="top")
ax.set_yticks(yv); ax.set_yticklabels(sm.model, fontsize=9)
ax.set_xlim(-0.4, 0.30); ax.set_xlabel("R²  (predicting the mean = 0)")
ax.set_title("Seven models, tuned, honestly validated — none beats a constant")
ax.legend(loc="lower left", fontsize=8.5); ax.grid(axis="y", visible=False)
EX["bakeoff"] = fig_to_b64(fig)
print("EX3, EX4 done")

EX3, EX4 done


In [4]:
# EX5 - skill x role heatmap (the "what it CAN do" hero)
SHOW = ["python", "sql", "r", "spark", "hadoop", "aws", "tableau", "excel",
        "tensorflow", "pytorch", "java", "sas"]
heat = pd.DataFrame({s: dedup.groupby("role_family")[f"skill_{s}"].mean() * 100 for s in SHOW})
heat = heat.reindex(["Data Analyst", "Data Engineer", "Data Scientist", "ML Engineer", "Other"]).astype(float)
fig, ax = plt.subplots(figsize=(11, 3.4))
im = ax.imshow(heat.values, cmap=mpl.colors.LinearSegmentedColormap.from_list("b", ["#FFFFFF", BLUE]),
               aspect="auto", vmin=0, vmax=100)
ax.set_xticks(range(len(SHOW))); ax.set_xticklabels(SHOW, rotation=45, ha="right", fontsize=8.5)
ax.set_yticks(range(len(heat))); ax.set_yticklabels(heat.index, fontsize=9)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.values[i, j]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8,
                color="white" if v > 55 else NAVY)
ax.set_title("Skill demand separates cleanly by role — the structure salary lacks")
ax.grid(False); fig.colorbar(im, ax=ax, shrink=0.8, label="% of postings")
EX["skills"] = fig_to_b64(fig)

# EX6 - tagger confusion + per-class F1
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report
CLS = ["Data Analyst", "Data Engineer", "Data Scientist", "ML Engineer", "Other"]
pipe = Pipeline([("tf", TfidfVectorizer(max_features=3000, stop_words="english", min_df=3,
                                        ngram_range=(1, 2), sublinear_tf=True)),
                 ("c", LinearSVC(class_weight="balanced", random_state=42))])
pred = cross_val_predict(pipe, dedup["Job Description"], dedup.role_family.astype(str),
                         cv=GroupKFold(5), groups=dedup.posting_id)
cm = confusion_matrix(dedup.role_family.astype(str), pred, labels=CLS)
rep = classification_report(dedup.role_family.astype(str), pred, labels=CLS, output_dict=True, zero_division=0)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4), gridspec_kw={"width_ratios": [1.05, 1]})
cmn = cm / cm.sum(1, keepdims=True)
a1.imshow(cmn, cmap=mpl.colors.LinearSegmentedColormap.from_list("b", ["#FFFFFF", BLUE]), vmin=0, vmax=1, aspect="auto")
a1.set_xticks(range(5)); a1.set_xticklabels([c.replace(" ", "\n") for c in CLS], fontsize=8)
a1.set_yticks(range(5)); a1.set_yticklabels(CLS, fontsize=8.5)
for i in range(5):
    for j in range(5):
        a1.text(j, i, cm[i, j], ha="center", va="center", fontsize=9, color="white" if cmn[i, j] > 0.5 else NAVY)
a1.set_xlabel("predicted"); a1.set_ylabel("actual"); a1.set_title("Confusion matrix"); a1.grid(False)
f1s = [rep[c]["f1-score"] for c in CLS]
a2.barh(range(5), f1s, color=CATEGORICAL, height=0.6)
a2.axvline(K["tagger_macro_f1"], color=NAVY, ls="--", lw=1.3)
for i, f in enumerate(f1s):
    a2.text(f + 0.01, i, f"{f:.2f}", va="center", fontsize=8.5, color=NAVY)
a2.set_yticks(range(5)); a2.set_yticklabels(CLS, fontsize=8.5); a2.set_xlim(0, 1.05)
a2.set_xlabel("per-class F1"); a2.set_title(f"macro-F1 {K['tagger_macro_f1']:.2f} vs {K['tagger_baseline_f1']:.2f} baseline")
a2.grid(axis="y", visible=False)
EX["tagger"] = fig_to_b64(fig)
print("EX5, EX6 done")

EX5, EX6 done


In [5]:
# EX7 - association eta2 vs eps2 (why "nothing predicts salary" is a powered claim)
assoc = pd.read_csv(REPORTS / "_associations.csv").sort_values("eta2")
fig, ax = plt.subplots(figsize=(9, 3.6))
yv = np.arange(len(assoc))
ax.barh(yv, assoc.eta2, height=0.5, color=GREY, label="η² (biased upward by group count)")
ax.barh(yv, assoc.eps2, height=0.26, color=BLUE, label="ε² (bias-corrected)")
ax.scatter(assoc.eta2_detectable, yv, marker="|", s=300, color=AMBER, lw=2, label="80%-power floor")
ax.set_yticks(yv); ax.set_yticklabels(assoc.cut, fontsize=8)
ax.set_xlabel("share of salary variance explained")
ax.set_title("Bias-corrected, no job attribute clears its detection floor")
ax.legend(loc="lower right", fontsize=7.5); ax.grid(axis="y", visible=False)
EX["assoc"] = fig_to_b64(fig)

# EX8 - credential gating
cred = (dedup.groupby("role_family").degree_phd.mean() * 100).reindex(CLS)
fig, ax = plt.subplots(figsize=(7.5, 3.2))
ax.bar(range(5), cred, color=[BLUE if v > 40 else GREY for v in cred], width=0.6)
for i, v in enumerate(cred):
    ax.text(i, v + 1, f"{v:.0f}%", ha="center", fontsize=9, color=NAVY)
ax.set_xticks(range(5)); ax.set_xticklabels([c.replace(" ", "\n") for c in CLS], fontsize=8.5)
ax.set_ylabel("% demanding a PhD"); ax.set_ylim(0, 65)
ax.set_title("A PhD gates half of ML-Engineer postings — an access barrier")
ax.grid(axis="x", visible=False)
EX["cred"] = fig_to_b64(fig)
print(f"all {len(EX)} exhibits rendered")

all 8 exhibits rendered


## 7.2 Assemble the HTML

Single 760px column, answer panel with a blue left-border, Georgia serif governing-thought pull-quote,
4-up mono stat tiles, part dividers, white figure-cards with blue-bordered "So What" notes, a gold
recommendation callout, and a full appendix. Light/dark themes are token-driven via `:root` +
`prefers-color-scheme` + `:root[data-theme]` so the viewer's toggle wins in both directions.

In [6]:
CSS = """
:root{
  --ground:#f4f6f8; --surface:#ffffff; --ink:#051C2C; --muted:#5b6b78;
  --accent:#2251FF; --accent-soft:#eaf0ff; --teal:#00857C; --amber:#8A5E12;
  --grid:#E9ECEF; --rule:#d9e0e6;
  --serif:Georgia,'Times New Roman',serif;
  --sans:'Helvetica Neue',Arial,system-ui,'Segoe UI',sans-serif;
  --mono:ui-monospace,'SF Mono',Consolas,monospace;
}
@media (prefers-color-scheme:dark){:root{
  --ground:#051C2C; --surface:#0c2434; --ink:#e9eef2; --muted:#9fb2c0;
  --accent:#5a8dff; --accent-soft:#12324a; --amber:#E5B84A; --grid:#173142; --rule:#1c3b4f;
}}
:root[data-theme="light"]{
  --ground:#f4f6f8; --surface:#ffffff; --ink:#051C2C; --muted:#5b6b78;
  --accent:#2251FF; --accent-soft:#eaf0ff; --amber:#8A5E12; --grid:#E9ECEF; --rule:#d9e0e6;
}
:root[data-theme="dark"]{
  --ground:#051C2C; --surface:#0c2434; --ink:#e9eef2; --muted:#9fb2c0;
  --accent:#5a8dff; --accent-soft:#12324a; --amber:#E5B84A; --grid:#173142; --rule:#1c3b4f;
}
*{box-sizing:border-box}
body{margin:0;background:var(--ground);color:var(--ink);font-family:var(--sans);
  line-height:1.6;-webkit-font-smoothing:antialiased}
.wrap{max-width:760px;margin:0 auto;padding:56px 24px 96px}
.eyebrow{font-family:var(--mono);font-size:11px;letter-spacing:.18em;text-transform:uppercase;
  color:var(--muted);margin:0 0 10px}
h1{font-family:var(--serif);font-size:34px;line-height:1.18;font-weight:600;margin:0 0 8px;
  text-wrap:balance}
.sub{color:var(--muted);font-size:15px;margin:0 0 4px}
h2{font-size:22px;font-weight:700;margin:8px 0 4px;text-wrap:balance}
h3{font-size:16px;font-weight:700;margin:26px 0 6px}
p{margin:0 0 14px}
.answer{background:var(--surface);border-left:4px solid var(--accent);border-radius:6px;
  padding:26px 28px;margin:26px 0 8px;box-shadow:0 1px 2px rgba(0,0,0,.04)}
.scr{font-size:14.5px;margin:0 0 6px}
.scr b{font-family:var(--mono);font-size:11px;letter-spacing:.1em;text-transform:uppercase;
  color:var(--accent);display:inline-block;min-width:104px}
.gov{font-family:var(--serif);font-size:20px;line-height:1.4;color:var(--ink);
  background:var(--accent-soft);border-radius:6px;padding:18px 22px;margin:18px 0;font-style:italic}
.keylines{margin:14px 0;padding:0;counter-reset:kl}
.keylines li{list-style:none;position:relative;padding:8px 0 8px 40px;margin:0;
  border-top:1px solid var(--rule);font-size:14.5px}
.keylines li:before{counter-increment:kl;content:counter(kl);position:absolute;left:0;top:8px;
  width:26px;height:26px;background:var(--accent);color:#fff;border-radius:50%;
  font-family:var(--mono);font-size:13px;display:flex;align-items:center;justify-content:center;font-weight:600}
.rec{background:var(--surface);border:1px solid var(--rule);border-top:3px solid var(--amber);
  border-radius:6px;padding:20px 24px;margin:18px 0}
.rec .eyebrow{color:var(--amber)}
.tiles{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin:26px 0}
.tile{background:var(--surface);border:1px solid var(--rule);border-radius:6px;padding:16px 14px;text-align:center}
.tile .n{font-family:var(--mono);font-size:26px;font-weight:700;color:var(--accent);
  font-variant-numeric:tabular-nums;line-height:1}
.tile .l{font-size:11px;color:var(--muted);margin-top:8px;line-height:1.35}
.divider{border:0;border-top:2px solid var(--accent);margin:52px 0 0}
.divider+.eyebrow{margin-top:12px;color:var(--accent)}
.figcard{background:#fff;border:1px solid var(--rule);border-radius:8px;padding:16px;margin:18px 0 0}
.figcard img{width:100%;height:auto;display:block;border-radius:2px}
.sowhat{border-left:3px solid var(--accent);background:var(--accent-soft);border-radius:0 6px 6px 0;
  padding:12px 16px;margin:12px 0 8px;font-size:13.5px}
.sowhat b{color:var(--accent)}
.src{font-family:var(--mono);font-size:10.5px;color:var(--muted);margin:6px 0 0}
table{width:100%;border-collapse:collapse;margin:16px 0;font-size:13px}
th{font-family:var(--mono);font-size:10.5px;letter-spacing:.06em;text-transform:uppercase;
  text-align:left;color:var(--muted);border-bottom:1.5px solid var(--rule);padding:8px 10px}
td{border-bottom:1px solid var(--grid);padding:8px 10px;font-variant-numeric:tabular-nums}
tr.best td{background:var(--accent-soft)}
.appendix{margin-top:16px;font-size:13.5px;color:var(--muted)}
.appendix h3{color:var(--ink)}
code{font-family:var(--mono);font-size:12.5px;background:var(--accent-soft);padding:1px 5px;border-radius:3px}
.foot{margin-top:60px;padding-top:18px;border-top:1px solid var(--rule);
  font-family:var(--mono);font-size:11px;color:var(--muted)}
"""
print(f"CSS ready ({len(CSS)} chars)")

CSS ready (4496 chars)


In [7]:
def tile(n, l):
    return f'<div class="tile"><div class="n">{n}</div><div class="l">{l}</div></div>'

def figblock(img, sowhat, src):
    return (f'<div class="figcard"><img src="{img}" alt=""></div>'
            f'<div class="sowhat">{sowhat}</div><p class="src">{src}</p>')

SRC = "Source: Kaggle 'Uncleaned Data Science Jobs' (Glassdoor scrape, ~2020, US-centric). 672 postings / 492 unique."

HTML = f"""<div class="wrap">
<p class="eyebrow">Talent Analytics · Data-Source Review · Tier A</p>
<h1>This Glassdoor dataset cannot benchmark pay — but it can map the data-science role market</h1>
<p class="sub">A forensic review of the most widely used public DS-jobs dataset, for the compensation-review and job-architecture cycle.</p>

<div class="answer">
<p class="scr"><b>Situation</b> Talent teams need external salary benchmarks and role-spec guidance, and this 672-posting Glassdoor scrape is the most widely used public dataset for exactly that — its canonical use is a salary-prediction model, reproduced in hundreds of public notebooks.</p>
<p class="scr"><b>Complication</b> The salary field does not behave like a measurement. {K['n_distinct_salary_values']} distinct values cover {K['n_rows']} rows in {K['n_batches']} perfectly contiguous index blocks; {K['pct_multi_band']}% of repeated postings carry two or more different bands, up to ${K['max_spread_k']:.0f}K apart; and seniority carries a ${abs(K['seniority_delta_k']):.1f}K <i>discount</i>.</p>
<p class="scr"><b>Resolution</b> <code>Salary Estimate</code> is a search-query artifact — the scrape batch label, not the job's pay. The comp-benchmarking use must be <b>stopped, not caveated</b>. The file's genuine value is a role-and-skill market map, where role family is recoverable from description text at macro-F1 {K['tagger_macro_f1']:.2f}.</p>

<div class="gov">The only thing that predicts a posting's “salary” is which scrape returned it; the data that says nothing about pay says a great deal about roles and skills.</div>

<ol class="keylines">
<li><b>The salary field is a scrape artifact.</b> {K['n_batches']} salary bands fall in {K['n_batches']} unbroken blocks of rows (η²=1.000, permutation p&lt;0.0001), while company rating varies freely within those same blocks (η²={K['eta2_rating_batch']}) — ruling out an innocent sort.</li>
<li><b>Nothing predicts it, and no model beats a constant.</b> Every job attribute falls below its statistical detection floor; seven tuned models under honest validation top out at R²={K['salary_best_grouped_r2']:+.2f}, short of the pre-registered 0.20 bar.</li>
<li><b>The role-and-skill structure is real.</b> Tableau appears in 51% of analyst postings and 0% of ML-engineer postings; a PhD is demanded by {K['phd_gate_mle']:.0f}% of ML-engineer postings versus {K['phd_gate_analyst']:.0f}% of analyst postings.</li>
<li><b>One model ships.</b> Role family is recoverable from description text at macro-F1 {K['tagger_macro_f1']:.2f} ({K['tagger_accuracy']:.0%} accuracy) versus a {K['tagger_baseline_f1']:.2f} baseline — useful auto-tagging for a title field that is 50% one value.</li>
</ol>

<div class="rec">
<p class="eyebrow">Recommendation</p>
<p style="margin:0">Talent Analytics should (1) <b>remove this source from the compensation-benchmarking evidence base</b> before the next comp cycle; (2) <b>adopt it as a role-and-skill structure reference</b>, quoted at the posting level, never for pay; and (3) <b>ship the role-family tagger</b> against the {K['n_raw_titles']}-value title field, with class-aware thresholds and the vocabulary-drift monitor attached.</p>
</div>
</div>

<div class="tiles">
{tile(f"{K['n_distinct_salary_values']}", f"distinct salary values across {K['n_rows']} postings")}
{tile(f"{K['pct_multi_band']:.0f}%", "of repeated postings carry conflicting salaries")}
{tile(f"{K['salary_best_grouped_r2']:+.2f}", "best salary R² — worse than a constant")}
{tile(f"{K['tagger_macro_f1']:.2f}", "role-tagger macro-F1 (0.16 baseline)")}
</div>

<hr class="divider"><p class="eyebrow">Part I · The salary field is an artifact</p>
<h2>Thirty salary values fall in thirty unbroken blocks of rows</h2>
<p>If salary were a property of the job, its values would scatter through the file in collection order. Instead each of the {K['n_batches']} distinct values occupies one contiguous run of row indices — the fingerprint of {K['n_batches']} separate Glassdoor searches, each with its own salary filter, concatenated in order.</p>
{figblock(EX["hero"], f"<b>So What:</b> the file is {K['n_batches']} search queries stitched together; the salary column is a batch label denominated in dollars. A permutation test rejects chance at p&lt;0.0001 — a random assignment would produce ≈{K['perm_null_mean_runs']:.0f} blocks, not {K['observed_runs']}.", SRC)}

<h2>The identical posting sells for wildly different pay depending only on which search returned it</h2>
{figblock(EX["repeat"], f"<b>So What:</b> {K['n_multi_band']} of {K['n_repeated_postings']} repeated postings ({K['pct_multi_band']}%) carry two or more salary bands, median disagreement ${K['median_spread_k']}K, maximum ${K['max_spread_k']:.0f}K. A quantity that takes five values for one unchanged job is not measuring that job.", SRC)}

<h2>Salary is frozen within a batch while everything else varies freely — ruling out a simple sort</h2>
{figblock(EX["control"], f"<b>So What:</b> the obvious objection is that the vendor sorted rows by pay. That would have moved every column; instead salary is constant within batch (η²=1.000) while rating (η²={K['eta2_rating_batch']}, p={K['rating_batch_p']:.2f}), posting length and company age vary freely. The artifact is specific to the target.", SRC)}

<hr class="divider"><p class="eyebrow">Part II · Nothing predicts it</p>
<h2>No job attribute survives a powered test for a salary effect</h2>
<p>A null result only counts when the test could have found something. Each attribute is shown with the effect its own sample size could detect at 80% power, and with the bias-corrected ε² that strips out the spurious variance many groups create by chance.</p>
{figblock(EX["assoc"], f"<b>So What:</b> the only cut that clears both its floor and FDR correction is estimate <i>provenance</i> — scrape metadata, not a job attribute. Every genuine attribute is null: seniority moves pay ${K['seniority_delta_k']:+.1f}K (d={K['seniority_d']:+.2f}) where d={K['seniority_mde_d']:.2f} was detectable, and rating correlates r={K['rating_salary_r']:+.3f}.", SRC)}

<h2>Seven tuned models, both protocols, and not one beats predicting the mean</h2>
{figblock(EX["bakeoff"], f"<b>So What:</b> the best honest score is R²={K['salary_best_grouped_r2']:+.2f} — below a constant, and under a third of the way to the pre-registered {0.20} bar. The random-split bars sit beside the honest ones: the 43% duplicate leakage buys nothing, because a repeated posting carries a different label. The rejection is now the verdict of a properly built model.", SRC)}

<table>
<tr><th>Salary model (grouped CV)</th><th>R² honest</th><th>R² leaky</th><th>MAE $K</th></tr>
{"".join(f'<tr class="{"best" if r.model=="MeanBaseline" else ""}"><td>{r.model}</td><td>{r.R2_grouped:+.3f}</td><td>{r.R2_random:+.3f}</td><td>{r.MAE_K:.1f}</td></tr>' for r in pd.read_csv(REPORTS/"_salary_models.csv").itertuples())}
</table>

<hr class="divider"><p class="eyebrow">Part III · What the file does show</p>
<h2>Skill demand separates cleanly by role — the structure salary lacks</h2>
{figblock(EX["skills"], "<b>So What:</b> the same file that carries no salary signal carries strong role signal. Tableau runs 51%→0% across the analyst-to-ML-engineer axis; Spark runs the other way. These are large, monotone, interpretable — the raw material for a skills taxonomy and job-spec guidance.", SRC.replace("672 postings / 492 unique","492 unique postings"))}

<h2>A PhD gates half of ML-engineer postings — an access barrier, not a skill signal</h2>
{figblock(EX["cred"], f"<b>So What:</b> a PhD is named in {K['phd_gate_mle']:.0f}% of ML-engineer postings versus {K['phd_gate_analyst']:.0f}% of analyst postings, while median stated experience is flat across roles. That is a hiring-design and access-equity finding: the ML-engineer entry gate is a credential, with documented disparate-impact risk.", SRC.replace("672 postings / 492 unique","492 unique postings"))}

<hr class="divider"><p class="eyebrow">Part IV · The model that ships</p>
<h2>Role family is recoverable from description text at macro-F1 {K['tagger_macro_f1']:.2f}</h2>
{figblock(EX["tagger"], f"<b>So What:</b> a linear text classifier recovers role family at macro-F1 {K['tagger_macro_f1']:.2f} ({K['tagger_accuracy']:.0%} accuracy) versus a {K['tagger_baseline_f1']:.2f} majority baseline — Data Scientist near-perfect (F1 {K['tagger_ds_f1']:.2f}). Confusions cluster on the Data-Engineer↔ML-Engineer boundary, a real vocabulary overlap. It needs class-aware thresholds before deployment.", SRC.replace("672 postings / 492 unique","492 unique postings"))}

<hr class="divider"><p class="eyebrow">Appendix · Methodology & limitations</p>
<div class="appendix">
<h3>Pipeline</h3>
<p>Six notebooks: ingestion (schema + batch-structure gate) → cleaning (sentinel audit, three-level dedup) → EDA → forensic diagnostic (permutation test, effect sizes, MDE, FDR, pre-registration) → modelling (7-family salary bake-off, role tagger, rating model) → this report. Seed 42 throughout; <code>GroupKFold</code> on posting identity for every model. Stack: pandas, scikit-learn, statsmodels, xgboost.</p>
<h3>Why the salary model was built at all</h3>
<p>The project owner asked for the canonical salary-prediction deliverable in full. It was built to its best — full feature set, seven families, tuned XGBoost (tuned R²={K['salary_tuned_xgb_r2']:+.2f}) — precisely so the rejection could not be attributed to a weak attempt. The acceptance test was <b>pre-registered before any model ran</b> (<code>reports/_preregistration.json</code>).</p>
<h3>The irreducible ceiling</h3>
<p>Because identical postings carry different bands, within-replicate variance caps R² at ≈{K['ceiling_replicated']:.2f} on the rows where the floor is observable — no feature set can exceed it. The report quotes {K['ceiling_replicated']:.2f}, not the optimistic {K['ceiling_all_rows']:.2f}.</p>
<h3>Fairness</h3>
<p>The tagger affects access to opportunities, so a protected-subgroup audit is mandatory — and <b>impossible</b>: the file has no applicant or demographic attribute. That impossibility caps any deployment recommendation. Runnable audits were run: tagger error concentrates in the smallest role families (up to {K['tagger_worst_class_error']:.0%}), and credential gating is a documented access barrier.</p>
<h3>Monitoring (specification, untested)</h3>
<p>PSI on TF-IDF vocabulary and predicted role mix; macro-F1 on a labelled monthly sample. The expected failure mode is dated vocabulary — this ~2020 corpus contains no “LLM”, “RAG”, or “prompt engineering”, so the tagger degrades on modern postings by construction.</p>
<h3>Limitations</h3>
<p>Salary figures are Glassdoor <b>estimates</b>, never observed pay (20 rows are employer-supplied on a different basis). There is <b>no date field</b>, so nothing here is a trend and the ~2020 vintage is inferred. n={K['n_dedup']} unique postings is small, so state- and sector-level cuts are directional only. The scrape is US-centric and DS-title-biased by construction. <code>role_family</code> is a keyword map authored here, so the tagger's ceiling is limited by label noise. No causal claim is available for any association.</p>
</div>

<p class="foot">Tier A · McKinsey communication standard · notebooks 01–06 · seed 42 · reproducible from <code>Uncleaned_DS_jobs.csv</code></p>
</div>"""
print(f"HTML body assembled ({len(HTML):,} chars)")

HTML body assembled (483,930 chars)


In [8]:
PAGE = f"""<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>DS Jobs — Data-Source Review</title>
<style>{CSS}</style></head>
<body>{HTML}</body></html>"""

out = REPORTS / "final_report.html"
out.write_text(PAGE, encoding="utf-8")
kb = out.stat().st_size / 1024
print(f"wrote {out.relative_to(ROOT)}  ({kb:.0f} KB, self-contained)")
assert 'src="http' not in PAGE and 'href="http' not in PAGE and "@import" not in PAGE, "external ref found"
assert PAGE.count("data:image/png") == 8, "expected 8 embedded figures"
print("no external requests; 8 figures embedded as base64")

wrote reports\final_report.html  (477 KB, self-contained)
no external requests; 8 figures embedded as base64


## 7.3 One-Page Test

**Can a senior stakeholder decide from page 1 alone?** The answer panel carries the SCR, the Governing
Thought, four quantified Key Lines, and a specific recommendation naming who does what by when. Yes.

## Stage 7 — Gate Checklist (McKinsey Standard)

- [x] **SCR finalized** — evidence-backed Situation, Complication, Resolution
- [x] **Governing Thought** — one sentence answering the business question
- [x] **Key Lines MECE** — 4 arguments mapped to branches A / A-B / C / D, no overlap, no gap
- [x] **Executive summary passes the One-Page Test**
- [x] **All titles are Action Titles** — every section states its insight
- [x] **Every exhibit has a "So What"** in plain business language
- [x] **Vertical logic holds** — the section titles alone tell the story
- [x] **No orphan findings** — every exhibit serves a Key Line; methodology in the appendix
- [x] **Recommendation is specific** — who (Talent Analytics), what (stop / adopt / ship), by when (next comp cycle)
- [x] **Self-contained HTML** — all CSS inline, 8 figures base64, zero external requests
- [x] **Light/dark theme-aware** — `:root` + `prefers-color-scheme` + `:root[data-theme]`
- [x] **Reproducible** — seed 42, pinned deps, numbers sourced from `_key_figures.json`

**Next:** `README.md` (project + root row), then ship notebooks + HTML + README to GitHub.